In [1]:
import jax
import jax.numpy as jnp
import qcsys as qs
import scqubits
import numpy as np
import qutip

jax.config.update('jax_platform_name', 'cpu')
# jax.config.update('jax_enable_x64', True)
def transform_op_into_dressed_basis_jax(op_matrix, dressed_evecs):
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - dressed_evecs: A 2D JAX array representing the dressed eigenvectors.

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    S = jnp.conj(dressed_evecs)
    data = jnp.dot(S, jnp.dot(op_matrix, S.T.conj()))
    return data

# fluxonium

In [2]:

qsf = qs.Fluxonium.create(
    25,
    {"Ej": 2.7, "Ec": 0.6, "El": 0.13, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
)

scf = scqubits.Fluxonium(EJ=2.7,
                        EC=0.6,
                        EL=0.13,
                        flux=0,cutoff=100,
                        truncated_dim=25)


np.array_equal(np.array(scf.n_operator())[:25,:25] , np.array(qsf.ops['n'].data,dtype = 'complex128')), \
    np.allclose(np.array(qutip.Qobj(scf.hamiltonian()).tidyup()), np.array(qsf.get_H_full().data,dtype = 'complex128'),atol=1e-08)


(False, True)

# transmon

In [3]:
qst_sc = qs.SingleChargeTransmon.create(
    N = 4,
    params = {"Ej": 40, "Ec": 0.5,"ng":0.0},
    N_max_charge=30
)

sct = scqubits.Transmon(
    EJ=40,
    EC=0.5,
    ng=0.0,
    ncut=30,
    truncated_dim = 4
    )

np.array_equal(np.array(sct.n_operator()),np.array(qst_sc.linear_ops['n'].data)), \
    np.array_equal(np.array(sct.hamiltonian()),np.array(qst_sc.get_H_full().data))

(True, True)

# hilbertspace: Make bare hamiltonian the same 

In [4]:
import jaxquantum as jqt
devices = [qsf, qst_sc]
f_indx = 0
t_indx = 1
Ns = [device.N for device in devices]
tn = qs.promote(qst_sc.common_ops()["n"], t_indx, Ns)
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
g_tf = 0.2
system = qs.System.create(devices, couplings=[
    # g_tf * tn @ fn
])
system.params["g_tf"] = g_tf
Es, kets = system.calculate_eig_linear()


hilbertspace = scqubits.HilbertSpace([scf, sct])
hilbertspace.add_interaction(g_strength=g_tf,op1=scf.n_operator, op2=sct.n_operator,add_hc=False)
evals, evecs = hilbertspace.hamiltonian().eigenstates()


# Tmon part bare H
evals = sct.eigenvals(evals_count=sct.truncated_dim)
scq_tmon_bare = np.array(hilbertspace.diag_hamiltonian(sct, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_tmon_bare = np.array(system.promote(qst_sc.get_H(), 1).data)

# Fluxonium part bare H
evals = scf.eigenvals(evals_count=scf.truncated_dim)
scq_f_bare = np.array(hilbertspace.diag_hamiltonian(scf, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_f_bare = np.array(system.promote(qsf.get_H(), 0).data)


np.allclose(scq_f_bare,qcs_f_bare), \
    np.allclose(scq_tmon_bare,qcs_tmon_bare), \
        np.allclose(np.array(system.get_H_bare().data), np.array(hilbertspace.bare_hamiltonian()))

(True, True, True)

# hilbertspace: Make interaction hamiltonian the same

In [5]:
system = qs.System.create(devices, couplings=[
    g_tf * tn @ fn
])

TypeError: incompatible dimensions [[25, 61], [25, 61]] and [[25, 4], [25, 4]]

In [ ]:
tn @ fn

TypeError: incompatible dimensions [[25, 61], [25, 61]] and [[25, 4], [25, 4]]

scqubits interaction hamiltonian is a sum over interaction terms
    interaction terms = g * id_wrapped_op1 * id_wrapped_op2

where id_wrapped_op = spec_utils.identity_wrap(
                    operator,
                    subsystem_list[subsys_index],
                    subsystem_list,
                    evecs=evecs,
                    op_in_eigenbasis=op_in_eigenbasis,
                )

basically spec_utils.identity_wrap tensors the convert_operator_to_qobj() transformed operator with identities

convert_operator_to_qobj() basically converts the operator into dressed eigenbasis

In [5]:
from scqubits.utils.spectrum_utils import *
operator = sct.n_operator()
subsystem = hilbertspace.subsystem_list[1]

evals_count = 4
hamiltonian_mat = subsystem.hamiltonian()
evals, evecs = sp.linalg.eigh(
    hamiltonian_mat,
    eigvals_only=False,
    subset_by_index=(0, evals_count - 1),
    check_finite=False,
)
evals, evecs = order_eigensystem(evals, evecs)
state_table = evecs
states_in_columns = state_table
mtable = states_in_columns.conj().T @ operator @ states_in_columns
subsys_operator =  qt.Qobj(mtable)


In [6]:
subsys_operator.tidyup()

Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 0.         -1.2310413   0.          0.03397347]
 [-1.2310413   0.         -1.70023304  0.        ]
 [ 0.         -1.70023304  0.          2.02624923]
 [ 0.03397347  0.          2.02624923  0.        ]]

In [7]:
qutip.Qobj(np.array(qst_sc.get_op_in_H_eigenbasis(qst_sc.linear_ops['n']).data))


Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 0.          1.2310413   0.         -0.03397347]
 [ 1.2310413   0.         -1.70023304  0.        ]
 [ 0.         -1.70023304  0.          2.02624923]
 [-0.03397347  0.          2.02624923  0.        ]]

In [32]:
import scipy as sp

In [25]:
evs, evecs = jnp.linalg.eigh(qst_sc._get_H_in_linear_basis().data)  # Hermitian
idxs_sorted = jnp.argsort(evs)
va =  evs[idxs_sorted]
ve = evecs[:, idxs_sorted]


v1, e1 = sp.linalg.eigh(qst_sc._get_H_in_linear_basis().data)  # Hermitian
idxs_sorted1 = np.argsort(v1)
v1 =  v1[idxs_sorted1]
e1 = e1[:, idxs_sorted1]

In [19]:
evals, evecs = sp.linalg.eigh(
    hamiltonian_mat,
    eigvals_only=False,
    # subset_by_index=(0, evals_count - 1),
    check_finite=False,
)
aa, bb = order_eigensystem(evals, evecs)


In [31]:
np.sum(np.abs(ve[:,:10]-e1[:,:10])>1e-4)

77